In [1]:
!pip install requests langchain langchain-google-genai langgraph langsmith pandas

In [2]:
# Force install the library if missing
try:
    import semanticscholar
except ImportError:
    !pip install semanticscholar
    print("Installed semanticscholar.")

import os
import logging
from semanticscholar import SemanticScholar

# --- Configure Logging ---
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger(__name__)

# --- API Keys Configuration ---
# Semantic Scholar API Key
SEMANTIC_SCHOLAR_API_KEY = "QyzAnc3la76icrOJH4oc72S3PG0c4DAOPO6sjb6e"

# LangSmith Configuration (Optional but good for tracing)
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "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"
os.environ["LANGCHAIN_PROJECT"] = "Research_Paper_Reviewer"

# Initialize Semantic Scholar Client
try:
    sch = SemanticScholar(api_key=SEMANTIC_SCHOLAR_API_KEY)
    logger.info("✅ Semantic Scholar Client Initialized successfully.")
except Exception as e:
    logger.error(f"Failed to initialize Semantic Scholar: {e}")

Installed semanticscholar.


In [3]:
import re
import json
import requests
from datetime import datetime

class ResearchAssistant:
    def __init__(self, data_dir="data"):
        # Set the target directory directly to data/pdfs
        self.data_dir = data_dir
        self.pdf_save_path = os.path.join(self.data_dir, "pdfs")

        # Ensure the directory exists
        os.makedirs(self.pdf_save_path, exist_ok=True)

    def sanitize_filename(self, text):
        """Creates a safe filename from a string."""
        s = re.sub(r'[\\/*?:"<>|]', "", text)
        return s.strip()[:100]

    def get_search_parameters(self):
        """Captures user input for dynamic research topics."""
        print("\n--- Research Configuration ---")
        topic = input("Enter research topic: ").strip()

        print("Optional Filters (press Enter to skip):")
        min_year = input("  Minimum Publication Year (e.g., 2020): ").strip()
        min_citations = input("  Minimum Citations (e.g., 10): ").strip()

        return {
            "topic": topic,
            "min_year": int(min_year) if min_year.isdigit() else 2015,
            "min_citations": int(min_citations) if min_citations.isdigit() else 0
        }

    def search_and_rank(self, params, fetch_limit=50, selection_limit=3):
        """Searches papers, filters them, and ranks them."""
        logger.info(f"Searching for papers on: {params['topic']}")
        try:
            results = sch.search_paper(
                params['topic'],
                limit=fetch_limit,
                fields=['title', 'authors', 'year', 'citationCount', 'openAccessPdf', 'url', 'abstract', 'venue']
            )
        except Exception as e:
            logger.error(f"Search API failed: {str(e)}")
            return []

        candidates = []
        current_year = datetime.now().year

        for paper in results:
            # Basic validation
            if not paper.title or not paper.year:
                continue
            if paper.year < params['min_year']:
                continue
            if (paper.citationCount or 0) < params['min_citations']:
                continue

            # --- Ranking Logic (from reference) ---
            recency_score = max(0, (paper.year - (current_year - 5))) * 2
            citations = paper.citationCount or 0
            impact_score = min(citations / 10, 20)

            # Check for PDF access
            pdf_url = paper.openAccessPdf['url'] if paper.openAccessPdf else None
            access_score = 50 if pdf_url else 0

            total_score = recency_score + impact_score + access_score

            candidates.append({
                "paperId": paper.paperId,
                "title": paper.title,
                "authors": [a['name'] for a in paper.authors] if paper.authors else [],
                "year": paper.year,
                "citations": citations,
                "venue": paper.venue,
                "url": paper.url,
                "pdf_url": pdf_url,
                "score": total_score
            })

        # Sort by score descending
        candidates.sort(key=lambda x: x['score'], reverse=True)

        # Filter only those with PDFs for the final selection if possible
        # (The scoring prioritizes them, but let's ensure we don't pick non-PDFs if PDFs exist)
        final_selection = [c for c in candidates if c['pdf_url']]

        # If not enough PDFs, fill with high scoring non-PDFs (though download will fail for them)
        if len(final_selection) < selection_limit:
            remaining = [c for c in candidates if not c['pdf_url']]
            final_selection.extend(remaining[:selection_limit - len(final_selection)])

        selected = final_selection[:selection_limit]
        logger.info(f"Screened {len(candidates)} papers. Selected top {len(selected)}.")
        return selected

    def download_pdfs(self, papers, topic):
        """Downloads PDFs directly into data/pdfs/."""
        logger.info(f"Starting PDF downloads into: {self.pdf_save_path}")

        successful_downloads = []
        for paper in papers:
            if not paper['pdf_url']:
                logger.warning(f"Skipping download (No Open Access URL): {paper['title']}")
                continue

            # Generate filename
            first_author = paper['authors'][0].split()[-1] if paper['authors'] else "Unknown"
            safe_title = self.sanitize_filename(paper['title'])
            filename = f"{paper['year']}_{first_author}_{safe_title}.pdf"

            # Save path: data/pdfs/filename.pdf
            save_path = os.path.join(self.pdf_save_path, filename)

            try:
                response = requests.get(paper['pdf_url'], timeout=30, headers={"User-Agent": "Mozilla/5.0"})
                if response.status_code == 200 and b"%PDF" in response.content[:20]:
                    with open(save_path, "wb") as f:
                        f.write(response.content)

                    paper['local_path'] = save_path
                    paper['download_status'] = "Success"
                    successful_downloads.append(paper)
                    logger.info(f"Downloaded: {filename}")
                else:
                    logger.warning(f"Invalid PDF content for: {paper['title']}")
            except Exception as e:
                logger.error(f"Download error for {paper['title']}: {e}")

        # Save Metadata in data/pdfs/ as well
        metadata_path = os.path.join(self.pdf_save_path, "dataset_metadata.json")
        with open(metadata_path, "w") as f:
            json.dump(papers, f, indent=4)

        logger.info(f"Process Complete. Metadata saved to: {metadata_path}")
        return successful_downloads

In [4]:
# Create an instance of the assistant
assistant = ResearchAssistant()

# Get user input
params = assistant.get_search_parameters()

if params['topic']:
    # Search and Rank
    top_papers = assistant.search_and_rank(params)

    if top_papers:
        print(f"\n--- Top {len(top_papers)} Papers Selected ---")
        for i, p in enumerate(top_papers, 1):
            print(f"{i}. [{p['year']}] {p['title']}")
            print(f"   Score: {p['score']} | Citations: {p['citations']}")
            print(f"   PDF: {'Available' if p['pdf_url'] else 'Not Available'}")
            print("-" * 30)

        # Download
        assistant.download_pdfs(top_papers, params['topic'])

        print(f"\n✅ Milestone 1 Complete. Check the '{assistant.pdf_save_path}' folder.")
    else:
        logger.warning("No suitable papers found matching criteria.")
else:
    logger.error("Topic is required.")


--- Research Configuration ---
Enter research topic: machine learning
Optional Filters (press Enter to skip):
  Minimum Publication Year (e.g., 2020): 2020
  Minimum Citations (e.g., 10): 5

--- Top 3 Papers Selected ---
1. [2024] Evaluation metrics and statistical tests for machine learning
   Score: 76 | Citations: 755
   PDF: Available
------------------------------
2. [2024] Leveraging large language models for predictive chemistry
   Score: 76 | Citations: 302
   PDF: Available
------------------------------
3. [2023] Understanding of Machine Learning with Deep Learning: Architectures, Workflow, Applications and Future Directions
   Score: 74 | Citations: 747
   PDF: Available
------------------------------



✅ Milestone 1 Complete. Check the 'data/pdfs' folder.


In [5]:
!pip install pymupdf4llm tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.3/72.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 79.1 MB/s eta 0:00:00


In [6]:
import pymupdf4llm
import glob

class TextExtractionModule:
    def __init__(self, data_dir="data"):
        self.pdf_dir = os.path.join(data_dir, "pdfs")
        self.processed_dir = os.path.join(data_dir, "processed")
        os.makedirs(self.processed_dir, exist_ok=True)

    def extract_text_from_pdf(self, pdf_path):
        """
        Extracts text from a single PDF using PyMuPDF4LLM.
        Returns the full markdown-formatted text.
        """
        try:
            # pymupdf4llm converts PDF to markdown, which is great for LLMs
            md_text = pymupdf4llm.to_markdown(pdf_path)
            return md_text
        except Exception as e:
            logger.error(f"Error extracting text from {pdf_path}: {e}")
            return None

    def process_all_pdfs(self):
        """
        Iterates through all PDFs in the data folder, extracts text,
        and saves it to the 'processed' directory.
        """
        pdf_files = glob.glob(os.path.join(self.pdf_dir, "*.pdf"))
        processed_data = []

        logger.info(f"Found {len(pdf_files)} PDFs to process.")

        for pdf_path in pdf_files:
            filename = os.path.basename(pdf_path)
            logger.info(f"Processing: {filename}")

            text = self.extract_text_from_pdf(pdf_path)

            if text:
                # Save extracted text to a .txt file for inspection/debugging
                txt_filename = filename.replace(".pdf", ".txt")
                save_path = os.path.join(self.processed_dir, txt_filename)

                with open(save_path, "w", encoding="utf-8") as f:
                    f.write(text)

                processed_data.append({
                    "filename": filename,
                    "text_path": save_path,
                    "full_text": text
                })

        return processed_data

# Run Extraction
extractor = TextExtractionModule()
extracted_docs = extractor.process_all_pdfs()

if extracted_docs:
    print(f"\n✅ Successfully processed {len(extracted_docs)} documents.")
    print(f"Sample content from first doc:\n{extracted_docs[0]['full_text'][:500]}...")

Consider using the pymupdf_layout package for a greatly improved page layout analysis.

✅ Successfully processed 2 documents.
Sample content from first doc:
## **OPEN**



www.nature.com/scientificreports

# **Evaluation metrics and statistical** **tests for machine learning**


    

**Research on different machine learning (ML) has become incredibly popular during the past few**
**decades. However, for some researchers not familiar with statistics, it might be difficult to understand**
**how to evaluate the performance of ML models and compare them with each other. Here, we**
**introduce the most common evaluation metrics used for the typical supe...


In [7]:
!pip install -U langchain-groq langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 490.2/490.2 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.7 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.6
    Uninstalling langchain-core-1.2.6:
      Successfully uninstalled langchain-core-1.2.6


In [8]:
import os
import json
import logging
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# --- CONFIGURATION ---
# 1. Set your specific API Key
os.environ["GROQ_API_KEY"] = "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"

# 2. Use the correct, active model (Fixes the 400 error)
MODEL_NAME = "llama-3.3-70b-versatile"

# Configure Logger
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class AnalysisModule:
    def __init__(self):
        try:
            self.llm = ChatGroq(
                groq_api_key=os.environ["GROQ_API_KEY"],
                model_name=MODEL_NAME,
                temperature=0.3
            )
            print(f"✅ Loaded Groq Model: {MODEL_NAME}")
        except Exception as e:
            print(f"❌ Error loading Groq model: {e}")

    def analyze_single_paper(self, text):
        """
        Extracts structured insights from a single paper.
        """
        # Truncate to ~30k chars to fit context window safely
        truncated_text = text[:30000]

        prompt = PromptTemplate(
            template="""
            You are an expert academic researcher. Analyze the following research paper text and extract the key information in a structured format.

            TEXT:
            {text}

            OUTPUT REQUIREMENTS:
            1. **Main Objective**: (1 sentence summary)
            2. **Methodology**: (Key techniques/algorithms used)
            3. **Key Findings**: (Bullet points of main results)
            4. **Limitations**: (Any stated weaknesses)

            Provide the output in clean Markdown.
            """,
            input_variables=["text"]
        )

        chain = prompt | self.llm | StrOutputParser()
        return chain.invoke({"text": truncated_text})

    def compare_papers(self, analyses):
        """
        Synthesizes findings across multiple papers.
        """
        combined_summaries = "\n\n".join([f"--- Paper {i+1} ---\n{a}" for i, a in enumerate(analyses)])

        prompt = PromptTemplate(
            template="""
            You are writing a systematic review. Below are summaries of {num_papers} research papers on the same topic.

            SUMMARIES:
            {summaries}

            TASK:
            Generate a "Cross-Paper Analysis" that includes:
            1. **Common Themes**: What do all these papers agree on?
            2. **Methodological Differences**: How do their approaches differ?
            3. **Contradictions/Gaps**: Are there conflicting results or missing areas?

            Keep it concise and high-level.
            """,
            input_variables=["num_papers", "summaries"]
        )

        chain = prompt | self.llm | StrOutputParser()
        return chain.invoke({"num_papers": len(analyses), "summaries": combined_summaries})

# --- EXECUTION ---
analyzer = AnalysisModule()
paper_analyses = []

# Check if we have documents from the previous step
if 'extracted_docs' in locals() and extracted_docs:
    print(f"\n--- Starting Analysis on {len(extracted_docs)} Papers ---")

    # 1. Analyze each paper individually
    for doc in extracted_docs:
        print(f"Analyzing: {doc['filename']}...")
        try:
            analysis = analyzer.analyze_single_paper(doc['full_text'])
            paper_analyses.append(analysis)
            print("✅ Analysis Complete.")
        except Exception as e:
            logger.error(f"Failed to analyze {doc['filename']}: {e}")
            print(f"❌ Error: {e}")

    # 2. Compare papers (if more than one)
    if len(paper_analyses) > 1:
        print("\n--- Starting Cross-Paper Comparison ---")
        try:
            cross_analysis = analyzer.compare_papers(paper_analyses)
            print("✅ Comparison Complete.")
        except Exception as e:
            print(f"❌ Comparison Error: {e}")
            cross_analysis = "Comparison failed."
    elif len(paper_analyses) == 1:
        cross_analysis = "Only one paper available. No cross-comparison possible."
    else:
        cross_analysis = "No papers were successfully analyzed."

    # 3. Store results
    final_analysis_results = {
        "individual_analyses": paper_analyses,
        "cross_analysis": cross_analysis
    }

    # Save to file
    with open("data/analysis_results.json", "w") as f:
        json.dump(final_analysis_results, f, indent=4)
    print("\n✅ Milestone 2 Complete. Results saved to 'data/analysis_results.json'")

    # Optional: Preview Results
    print("\n--- SYNTHESIS PREVIEW ---")
    print(cross_analysis[:500] + "...")

else:
    print("⚠️ No extracted documents found. Please run the Text Extraction cell (Cell 6) first.")

✅ Loaded Groq Model: llama-3.3-70b-versatile

--- Starting Analysis on 2 Papers ---
Analyzing: 2024_Rainio_Evaluation metrics and statistical tests for machine learning.pdf...
✅ Analysis Complete.
Analyzing: 2024_Jablonka_Leveraging large language models for predictive chemistry.pdf...
✅ Analysis Complete.

--- Starting Cross-Paper Comparison ---
✅ Comparison Complete.

✅ Milestone 2 Complete. Results saved to 'data/analysis_results.json'

--- SYNTHESIS PREVIEW ---
**Cross-Paper Analysis**

### 1. Common Themes
Both papers emphasize the importance of selecting suitable evaluation metrics and techniques for machine learning tasks. They also highlight the potential of advanced models, such as GPT-3, in achieving high performance in various tasks, including classification, regression, and materials science applications.

### 2. Methodological Differences
The main difference lies in their focus areas: Paper 1 provides a comprehensive overview of evaluation met...


In [9]:
import json
import os
from datetime import datetime
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# --- CONFIGURATION ---
# Ensure API Key is loaded
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = "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"

MODEL_NAME = "llama-3.3-70b-versatile"

class DraftGenerationModule:
    def __init__(self, data_dir="data"):
        self.data_dir = data_dir
        self.output_dir = os.path.join(data_dir, "drafts")
        os.makedirs(self.output_dir, exist_ok=True)

        # Load Analysis Data
        try:
            with open(os.path.join(data_dir, "analysis_results.json"), "r") as f:
                self.analysis_data = json.load(f)

            # Load Paper Metadata (for references)
            # We look for the metadata file saved in Milestone 1
            meta_path = os.path.join(data_dir, "pdfs", "dataset_metadata.json")
            if os.path.exists(meta_path):
                 with open(meta_path, "r") as f:
                    self.paper_metadata = json.load(f)
            else:
                self.paper_metadata = []
                print("⚠️ Warning: Paper metadata not found. References may be incomplete.")

        except FileNotFoundError:
            print("❌ Error: Analysis results not found. Please run Milestone 2 first.")
            self.analysis_data = None

        self.llm = ChatGroq(
            groq_api_key=os.environ["GROQ_API_KEY"],
            model_name=MODEL_NAME,
            temperature=0.4
        )

    def generate_abstract(self):
        """Generates a merged abstract (approx 100-150 words)."""
        print("✍️ Generating Abstract...")
        prompt = PromptTemplate(
            template="""
            Synthesize the following cross-paper analysis into a single, cohesive academic abstract.

            ANALYSIS CONTEXT:
            {cross_analysis}

            REQUIREMENTS:
            1. Start with a broad topic sentence.
            2. Summarize the collective methodology and key findings.
            3. End with the broader implication of this research.
            4. STRICT LIMIT: Keep it under 150 words.
            """,
            input_variables=["cross_analysis"]
        )
        chain = prompt | self.llm | StrOutputParser()
        return chain.invoke({"cross_analysis": self.analysis_data.get("cross_analysis", "")})

    def generate_methods_section(self):
        """Generates a detailed Methods Comparison section."""
        print("✍️ Generating Methods Section...")
        # Extract individual methodologies for the prompt
        individual_methods = "\n".join([
            f"Paper {i+1} Findings:\n{a}"
            for i, a in enumerate(self.analysis_data.get("individual_analyses", []))
        ])

        prompt = PromptTemplate(
            template="""
            Write a "Methodology Comparison" section for a review paper based on these extracted details.

            INDIVIDUAL PAPER DETAILS:
            {individual_methods}

            REQUIREMENTS:
            1. Compare the techniques used (e.g., specific algorithms, datasets, experimental setups).
            2. Highlight strengths and weaknesses of each approach.
            3. Group similar methods together if applicable.
            4. Use academic tone and formal language.
            """,
            input_variables=["individual_methods"]
        )
        chain = prompt | self.llm | StrOutputParser()
        return chain.invoke({"individual_methods": individual_methods})

    def generate_results_section(self):
        """Generates a Results Synthesis section."""
        print("✍️ Generating Results Section...")
        prompt = PromptTemplate(
            template="""
            Write a "Results and Discussion" section synthesizing the findings from these papers.

            CROSS-PAPER ANALYSIS:
            {cross_analysis}

            REQUIREMENTS:
            1. Discuss the key trends and agreements in results.
            2. Address any contradictions or diverging outcomes.
            3. Discuss the limitations identified across the studies.
            """,
            input_variables=["cross_analysis"]
        )
        chain = prompt | self.llm | StrOutputParser()
        return chain.invoke({"cross_analysis": self.analysis_data.get("cross_analysis", "")})

    def format_references_apa(self):
        """Formats the paper metadata into APA style citations."""
        print("📚 Formatting References (APA)...")
        references = []
        for p in self.paper_metadata:
            # Attempt to construct APA format: Author, A. A. (Year). Title. Venue.
            try:
                # Handle authors list
                authors = p.get('authors', [])
                if isinstance(authors, list):
                    if len(authors) > 0:
                        # Take just the first author for brevity if list is complex string,
                        # or format properly if it's a clean list
                        author_text = ", ".join(authors[:3]) + (" et al." if len(authors) > 3 else "")
                    else:
                        author_text = "Unknown Author"
                else:
                    author_text = str(authors)

                year = p.get('year', 'n.d.')
                title = p.get('title', 'Untitled')
                venue = p.get('venue', '')
                url = p.get('url', '')

                # APA-ish format string
                ref_entry = f"{author_text} ({year}). *{title}*. {venue}. Retrieved from {url}"
                references.append(ref_entry)
            except Exception as e:
                references.append(f"Error formatting citation for {p.get('title', 'paper')}")

        return "\n\n".join(references)

    def run_full_drafting_pipeline(self):
        """Runs all generation steps and compiles the final report."""
        if not self.analysis_data:
            return

        abstract = self.generate_abstract()
        methods = self.generate_methods_section()
        results = self.generate_results_section()
        refs = self.format_references_apa()

        # Compile Full Report
        full_report = f"""# Automated Systematic Review

## Abstract
{abstract}

---

## 1. Methodology Comparison
{methods}

## 2. Results and Discussion
{results}

---

## References
{refs}
        """

        # Save to file
        report_path = os.path.join(self.output_dir, "Final_Review_Draft.md")
        with open(report_path, "w") as f:
            f.write(full_report)

        print(f"\n✅ Draft Generated Successfully!")
        print(f"📄 Saved to: {report_path}")
        return full_report

# --- EXECUTE ---
writer = DraftGenerationModule()
final_report = writer.run_full_drafting_pipeline()

# Preview the Report
if final_report:
    from IPython.display import Markdown
    display(Markdown(final_report))

✍️ Generating Abstract...
✍️ Generating Methods Section...
✍️ Generating Results Section...
📚 Formatting References (APA)...

✅ Draft Generated Successfully!
📄 Saved to: data/drafts/Final_Review_Draft.md


# Automated Systematic Review

## Abstract
The integration of machine learning and advanced models has revolutionized various fields. A cross-paper analysis reveals that selecting suitable evaluation metrics and techniques is crucial for achieving high performance in machine learning tasks. Collectively, the papers provide a comprehensive overview of evaluation metrics and explore the potential of large language models, such as GPT-3, in tasks like classification and materials science applications. The analysis highlights a notable gap in evaluating large language models, underscoring the need for further research. Ultimately, this research implies that developing suitable evaluation metrics for advanced models is essential for harnessing their full potential and driving innovation in machine learning and related fields.

---

## 1. Methodology Comparison
## Methodology Comparison

This review paper examines the methodologies employed in two distinct research studies, each addressing unique aspects of machine learning. The first paper provides a comprehensive overview of evaluation metrics and statistical tests for various machine learning tasks, including classification, regression, image segmentation, object detection, and information retrieval. In contrast, the second paper explores the potential of fine-tuning large language models, specifically GPT-3, for tasks in chemistry and materials science.

### Evaluation Metrics and Statistical Tests

The first paper utilizes a range of evaluation metrics, such as accuracy, sensitivity, specificity, precision, recall, F1-score, Cohen's kappa, and Matthews' correlation coefficient for classification tasks. For regression tasks, metrics including mean absolute error, mean squared error, and Pearson's correlation coefficient are employed. Additionally, the paper discusses metrics like Dice score, Jaccard index, and Intersection over Union for image segmentation tasks, as well as average precision and mean average precision for object detection tasks. Statistical tests, such as paired t-test and resampling procedures, are used to compare model performance. This approach highlights the importance of selecting suitable evaluation metrics and statistical tests for comparing model performance, demonstrating a strength in providing a comprehensive framework for evaluating machine learning models.

### Fine-Tuning of Large Language Models

In contrast, the second paper focuses on fine-tuning GPT-3 models using task-specific datasets, in-context learning, and parameter-efficient fine-tuning techniques. This approach enables GPT-3 models to perform comparably to or even outperform conventional machine learning techniques, particularly in low-data limit scenarios. The use of fine-tuning and in-context learning allows for adaptability and flexibility in addressing various tasks in chemistry and materials science. However, this method requires significant amounts of data for regression tasks and may not be optimized for certain applications, such as those requiring high accuracy.

### Comparison of Techniques

A comparison of the techniques used in both papers reveals distinct strengths and weaknesses. The first paper provides a broad framework for evaluating machine learning models, offering a comprehensive overview of evaluation metrics and statistical tests. However, this approach may not be exhaustive, and the choice of evaluation metric and statistical test depends on the specific problem and data. The second paper demonstrates the potential of fine-tuning large language models for specific tasks, but may be limited by the requirement for significant amounts of data and potential biases in the models.

### Grouping of Similar Methods

Both papers can be grouped into the category of machine learning methodologies, with the first paper focusing on evaluation metrics and statistical tests, and the second paper exploring the fine-tuning of large language models. Within this category, the first paper can be further grouped with other studies that provide comprehensive overviews of evaluation metrics, while the second paper can be grouped with other research that investigates the application of large language models to specific tasks.

### Conclusion

In conclusion, the methodologies employed in the two papers demonstrate distinct approaches to addressing machine learning tasks. The first paper provides a comprehensive framework for evaluating machine learning models, while the second paper explores the potential of fine-tuning large language models for specific tasks. By comparing and contrasting these approaches, researchers can gain a deeper understanding of the strengths and weaknesses of each methodology, ultimately informing the development of more effective machine learning models.

## 2. Results and Discussion
## Results and Discussion

This analysis synthesizes the findings from two papers, highlighting common themes, methodological differences, and gaps in the existing literature. The results reveal key trends and agreements in the importance of evaluation metrics and the potential of advanced models, such as GPT-3, in machine learning tasks.

A key trend observed across both papers is the emphasis on selecting suitable evaluation metrics and techniques for machine learning tasks. Both papers agree on the significance of evaluation metrics in assessing the performance of machine learning models, including classification, regression, and materials science applications. This agreement underscores the need for careful consideration of evaluation metrics in machine learning research. Furthermore, the papers concur on the potential of advanced models, such as GPT-3, in achieving high performance in various tasks, demonstrating the promise of large language models in machine learning.

In terms of methodological differences, the papers focus on distinct aspects of machine learning. Paper 1 provides a comprehensive overview of evaluation metrics and statistical tests for various machine learning tasks, whereas Paper 2 explores the application of large language models (GPT-3) in chemistry and materials science. This difference in focus areas highlights the breadth of machine learning research, from foundational evaluation metrics to specialized applications.

Regarding contradictions or diverging outcomes, the analysis reveals no direct contradictions between the two papers. However, a notable gap is identified in the discussion on evaluation metrics used for large language models like GPT-3. Paper 1 provides a thorough overview of evaluation metrics, but it does not cover the specific challenges of evaluating large language models. This gap highlights the need for further research on developing and applying suitable evaluation metrics for advanced models like GPT-3.

The limitations identified across the studies include the lack of discussion on evaluation metrics for large language models in Paper 2. This limitation underscores the need for more research on evaluating advanced models, which is crucial for assessing their performance and potential applications. Additionally, the analysis highlights the importance of considering the specific challenges of evaluating large language models, which may require specialized evaluation metrics and techniques.

In conclusion, the results of this analysis demonstrate the importance of evaluation metrics and advanced models in machine learning research. The agreement on the significance of evaluation metrics and the potential of large language models highlights key trends in the field. The identified gap in the discussion on evaluation metrics for large language models underscores the need for further research, and the limitations identified across the studies emphasize the importance of careful consideration of evaluation metrics in machine learning research. Future studies should address these gaps and limitations to advance the field of machine learning.

---

## References
O. Rainio, J. Teuho, R. Klén (2024). *Evaluation metrics and statistical tests for machine learning*. Scientific Reports. Retrieved from https://www.semanticscholar.org/paper/4a2b5058000d567036fc9aaad75bbf71cce53002

K. Jablonka, P. Schwaller, Andres Ortega‐Guerrero et al. (2024). *Leveraging large language models for predictive chemistry*. Nature Machine Intelligence. Retrieved from https://www.semanticscholar.org/paper/cba71f24d5fd25ade9e60ef38df282aa91de5b80

Mohammad Mustafa Taye (2023). *Understanding of Machine Learning with Deep Learning: Architectures, Workflow, Applications and Future Directions*. De Computis. Retrieved from https://www.semanticscholar.org/paper/df70977e0347b76fb049c17c3956f643bcb43a55
        

In [11]:
!pip install streamlit langchain-groq langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 95.8 MB/s eta 0:00:00


In [17]:
%%writefile app.py
"""
AI System to Automatically Review and Summarize Research Papers
Milestone 4: Review, Refinement Cycle & Streamlit UI
- Quality assessment and revision suggestions
- Interactive Streamlit interface
- Critique/Revise functionality
- Final report generation
"""

import os
import json
import logging
from datetime import datetime
from typing import Dict, List, Optional, Tuple
import streamlit as st
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ==================== CONFIGURATION ====================
class Config:
    """Configuration for Milestone 4"""
    # Use absolute paths to avoid confusion in Colab
    BASE_DIR = os.path.abspath(os.getcwd())
    DATA_DIR = os.path.join(BASE_DIR, "data")
    DRAFTS_DIR = os.path.join(DATA_DIR, "drafts")
    FINAL_DIR = os.path.join(DATA_DIR, "final")
    LOGS_DIR = os.path.join(BASE_DIR, "logs")

    # Input files
    DRAFT_REPORT = os.path.join(DRAFTS_DIR, "Final_Review_Draft.md")
    ANALYSIS_RESULTS = os.path.join(DATA_DIR, "analysis_results.json")

    # Output files
    QUALITY_REPORT = os.path.join(FINAL_DIR, "quality_assessment.json")
    REVISION_LOG = os.path.join(FINAL_DIR, "revision_history.json")
    FINAL_REPORT = os.path.join(FINAL_DIR, "Final_Systematic_Review.md")

    # API Configuration
    GROQ_API_KEY = "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"
    MODEL_NAME = "llama-3.3-70b-versatile"

    @staticmethod
    def setup_directories():
        """Create necessary directories"""
        os.makedirs(Config.FINAL_DIR, exist_ok=True)
        os.makedirs(Config.LOGS_DIR, exist_ok=True)

# ==================== LOGGING SETUP ====================
class LoggerSetup:
    """Configure logging for Milestone 4"""

    @staticmethod
    def setup_logger(name: str = "Milestone4") -> logging.Logger:
        logger = logging.getLogger(name)
        logger.setLevel(logging.INFO)
        if not logger.handlers:
            os.makedirs(Config.LOGS_DIR, exist_ok=True)

            log_file = os.path.join(Config.LOGS_DIR, f"milestone4_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log")
            file_handler = logging.FileHandler(log_file, encoding='utf-8')
            file_handler.setLevel(logging.DEBUG)

            console_handler = logging.StreamHandler()
            console_handler.setLevel(logging.INFO)

            formatter = logging.Formatter(
                '%(asctime)s - %(levelname)s - %(message)s',
                datefmt='%H:%M:%S'
            )
            file_handler.setFormatter(formatter)
            console_handler.setFormatter(formatter)

            logger.addHandler(file_handler)
            logger.addHandler(console_handler)

        return logger

# ==================== CORE LOGIC MODULES ====================
class QualityAssessmentModule:
    def __init__(self, logger: Optional[logging.Logger] = None):
        self.logger = logger or logging.getLogger(__name__)
        os.environ["GROQ_API_KEY"] = Config.GROQ_API_KEY
        self.llm = ChatGroq(
            groq_api_key=Config.GROQ_API_KEY,
            model_name=Config.MODEL_NAME,
            temperature=0.2
        )

    def assess_quality(self, draft_text: str) -> Dict:
        self.logger.info("📋 Conducting quality assessment...")
        prompt = PromptTemplate(
            template="""
            You are an expert academic reviewer. Evaluate this systematic review draft on multiple criteria.

            DRAFT TEXT:
            {draft_text}

            EVALUATION CRITERIA (Rate 0-10 for each):
            1. **Coherence**: Does the review flow logically?
            2. **Comprehensiveness**: Are all major aspects covered?
            3. **Accuracy**: Are technical details accurate?
            4. **Clarity**: Is the writing clear?
            5. **Academic Rigor**: Does it meet academic standards?

            OUTPUT FORMAT (JSON):
            {{
              "coherence_score": <0-10>,
              "comprehensiveness_score": <0-10>,
              "accuracy_score": <0-10>,
              "clarity_score": <0-10>,
              "rigor_score": <0-10>,
              "overall_score": <average>,
              "strengths": ["strength 1", ...],
              "weaknesses": ["weakness 1", ...],
              "critical_issues": ["issue 1", ...] or [],
              "recommendations": ["recommendation 1", ...]
            }}
            Provide ONLY the JSON output.
            """,
            input_variables=["draft_text"]
        )
        chain = prompt | self.llm | StrOutputParser()
        try:
            result = chain.invoke({"draft_text": draft_text[:15000]})
            # Clean markdown code blocks if present
            result = result.strip().replace("```json", "").replace("```", "")
            assessment = json.loads(result)
            self.logger.info(f" ✅ Overall Quality Score: {assessment.get('overall_score', 0)}/10")
            return assessment
        except Exception as e:
            self.logger.error(f"Error in quality assessment: {e}")
            return {"overall_score": 0, "error": str(e)}

class RevisionModule:
    def __init__(self, logger: Optional[logging.Logger] = None):
        self.logger = logger or logging.getLogger(__name__)
        os.environ["GROQ_API_KEY"] = Config.GROQ_API_KEY
        self.llm = ChatGroq(
            groq_api_key=Config.GROQ_API_KEY,
            model_name=Config.MODEL_NAME,
            temperature=0.5
        )

    def generate_revision_suggestions(self, draft_text: str, assessment: Dict) -> List[str]:
        self.logger.info("💡 Generating revision suggestions...")
        weaknesses = assessment.get('weaknesses', [])
        critical_issues = assessment.get('critical_issues', [])
        issues_text = "\n".join(weaknesses + critical_issues)

        prompt = PromptTemplate(
            template="""
            Based on the quality assessment, provide 5-7 specific, actionable revision suggestions.
            ISSUES: {issues}
            OUTPUT FORMAT: A numbered list of suggestions (1-7 items).
            """,
            input_variables=["issues"]
        )
        chain = prompt | self.llm | StrOutputParser()
        result = chain.invoke({"issues": issues_text})
        suggestions = [line.lstrip('0123456789.-•) ').strip() for line in result.strip().split('\n') if line.strip()]
        return suggestions[:7]

    def apply_revisions(self, draft_text: str, suggestions: List[str]) -> str:
        self.logger.info("✏️ Applying revisions...")
        suggestions_text = "\n".join(f"{i+1}. {s}" for i, s in enumerate(suggestions))
        prompt = PromptTemplate(
            template="""
            Revise the following academic review based on these suggestions.
            ORIGINAL TEXT: {draft_text}
            SUGGESTIONS: {suggestions}
            OUTPUT: Provide only the revised text.
            """,
            input_variables=["draft_text", "suggestions"]
        )
        chain = prompt | self.llm | StrOutputParser()
        revised_text = chain.invoke({"draft_text": draft_text[:20000], "suggestions": suggestions_text})
        return revised_text.strip()

class FinalReportGenerator:
    def __init__(self, logger: Optional[logging.Logger] = None):
        self.logger = logger or logging.getLogger(__name__)

    def generate_final_report(self, revised_draft: str, assessment: Dict) -> str:
        self.logger.info("📄 Generating final report...")
        report_parts = [
            "# SYSTEMATIC LITERATURE REVIEW",
            f"**Generated**: {datetime.now().strftime('%B %d, %Y')}",
            f"**Quality Score**: {assessment.get('overall_score', 'N/A')}/10",
            "="*70,
            "## QUALITY METRICS",
            f"- **Coherence**: {assessment.get('coherence_score', 0)}/10",
            f"- **Comprehensiveness**: {assessment.get('comprehensiveness_score', 0)}/10",
            f"- **Accuracy**: {assessment.get('accuracy_score', 0)}/10",
            f"- **Clarity**: {assessment.get('clarity_score', 0)}/10",
            "="*70,
            revised_draft,
            "="*70,
            "## APPENDIX: QUALITY ASSESSMENT",
            "### Strengths",
            "\n".join(f"- {s}" for s in assessment.get('strengths', [])),
            "### Recommendations",
            "\n".join(f"- {r}" for r in assessment.get('recommendations', []))
        ]
        final_report = '\n\n'.join(report_parts)
        with open(Config.FINAL_REPORT, 'w', encoding='utf-8') as f:
            f.write(final_report)
        return final_report

class RevisionHistoryTracker:
    def __init__(self):
        self.history = []
        if os.path.exists(Config.REVISION_LOG):
            with open(Config.REVISION_LOG, 'r') as f:
                self.history = json.load(f)

    def add_revision(self, version: int, assessment: Dict, suggestions: List[str]):
        entry = {
            "version": version,
            "timestamp": datetime.now().isoformat(),
            "quality_score": assessment.get("overall_score", 0),
            "suggestions_applied": suggestions
        }
        self.history.append(entry)
        with open(Config.REVISION_LOG, 'w') as f:
            json.dump(self.history, f, indent=4)

# ==================== STREAMLIT UI HELPERS ====================

@st.cache_resource
def get_processing_modules():
    """Initialize processing modules once and cache them"""
    Config.setup_directories()
    logger = LoggerSetup.setup_logger()
    return (
        QualityAssessmentModule(logger),
        RevisionModule(logger),
        RevisionHistoryTracker(),
        FinalReportGenerator(logger),
        logger
    )

def load_draft_from_file():
    """Helper to read the draft file safely"""
    if os.path.exists(Config.DRAFT_REPORT):
        with open(Config.DRAFT_REPORT, 'r', encoding='utf-8') as f:
            return f.read()
    return None

def init_session_state():
    """Initialize session state variables, auto-loading draft if possible"""
    if 'current_draft' not in st.session_state:
        # AUTO-LOAD LOGIC: Try to load immediately on startup
        loaded_text = load_draft_from_file()
        st.session_state.current_draft = loaded_text if loaded_text else ""

    if 'current_assessment' not in st.session_state:
        st.session_state.current_assessment = {}
    if 'current_suggestions' not in st.session_state:
        st.session_state.current_suggestions = []
    if 'version' not in st.session_state:
        st.session_state.version = 1
    if 'history' not in st.session_state:
        st.session_state.history = []

# ==================== STREAMLIT MAIN APP ====================

def main():
    st.set_page_config(page_title="AI Research Reviewer", page_icon="🔬", layout="wide")

    # Initialize Core Logic
    qa_module, rev_module, hist_tracker, rep_gen, logger = get_processing_modules()
    init_session_state()

    st.title("🔬 AI Research Paper Review System")
    st.markdown("### Milestone 4: Quality Assessment & Revision")

    # Sidebar for Status and History
    with st.sidebar:
        st.header("Project Status")
        if st.session_state.current_draft:
            st.success("✅ Draft Loaded")
        else:
            st.warning("⚠️ No Draft Found")
            st.caption(f"Looking in: {Config.DRAFT_REPORT}")

        st.divider()
        st.header("Revision History")
        if os.path.exists(Config.REVISION_LOG):
             with open(Config.REVISION_LOG, 'r') as f:
                hist_data = json.load(f)
                for entry in hist_data:
                    st.text(f"v{entry['version']}: Score {entry['quality_score']}/10")

    # Main Tabs
    tab1, tab2, tab3, tab4 = st.tabs(["📄 Load & View", "📊 Assess Quality", "✏️ Apply Revisions", "📑 Final Report"])

    # --- TAB 1: LOAD DRAFT ---
    with tab1:
        st.subheader("Step 1: Review Current Draft")
        col1, col2 = st.columns([1, 4])

        with col1:
            # Changed to "Reload" since we auto-load now
            if st.button("🔄 Reload Draft File", type="secondary"):
                loaded_text = load_draft_from_file()
                if loaded_text:
                    st.session_state.current_draft = loaded_text
                    st.toast("Draft reloaded successfully!", icon="✅")
                    st.rerun()
                else:
                    st.error(f"File not found at: {Config.DRAFT_REPORT}")
                    st.info("Please ensure Milestone 3 completed successfully.")

        with col2:
            # Using session_state directly for value to ensure updates stick
            text_area = st.text_area(
                "Draft Content (You can edit manually here)",
                value=st.session_state.current_draft,
                height=600,
                key="draft_editor"
            )
            # Sync manual edits back to state
            if text_area != st.session_state.current_draft:
                st.session_state.current_draft = text_area

    # --- TAB 2: ASSESS QUALITY ---
    with tab2:
        st.subheader("Step 2: Assess Quality")

        if st.button("🔍 Run Quality Assessment", type="primary"):
            if not st.session_state.current_draft:
                st.error("Please load a draft first (Check Tab 1).")
            else:
                with st.spinner("Analyzing draft coherence, accuracy, and rigor..."):
                    assessment = qa_module.assess_quality(st.session_state.current_draft)
                    st.session_state.current_assessment = assessment

                    # Generate suggestions immediately after assessment
                    suggestions = rev_module.generate_revision_suggestions(
                        st.session_state.current_draft, assessment
                    )
                    st.session_state.current_suggestions = suggestions
                st.toast("Assessment Complete!", icon="🎉")

        # Display Results if available
        if st.session_state.current_assessment:
            res = st.session_state.current_assessment

            # Metric Cards
            m1, m2, m3, m4, m5 = st.columns(5)
            m1.metric("Overall Score", f"{res.get('overall_score', 0)}/10")
            m2.metric("Coherence", f"{res.get('coherence_score', 0)}/10")
            m3.metric("Accuracy", f"{res.get('accuracy_score', 0)}/10")
            m4.metric("Clarity", f"{res.get('clarity_score', 0)}/10")
            m5.metric("Rigor", f"{res.get('rigor_score', 0)}/10")

            # Details
            col_l, col_r = st.columns(2)
            with col_l:
                st.markdown("### ✅ Strengths")
                for s in res.get('strengths', []):
                    st.success(s)

            with col_r:
                st.markdown("### ⚠️ Weaknesses")
                for w in res.get('weaknesses', []):
                    st.warning(w)

            st.markdown("### 💡 Recommendations")
            for r in res.get('recommendations', []):
                st.info(r)

    # --- TAB 3: APPLY REVISIONS ---
    with tab3:
        st.subheader("Step 3: Apply Revisions")

        if not st.session_state.current_suggestions:
            st.info("Run the Quality Assessment in Step 2 to generate suggestions.")
        else:
            st.markdown("### Suggested Improvements")

            # Editable suggestions
            suggestions_text = st.text_area(
                "Edit suggestions before applying (one per line)",
                value="\n".join(st.session_state.current_suggestions),
                height=150
            )

            if st.button("⚡ Apply AI Revisions", type="primary"):
                with st.spinner("AI is rewriting the draft based on your suggestions..."):
                    # Parse editable suggestions back to list
                    clean_suggestions = [s.strip() for s in suggestions_text.split('\n') if s.strip()]

                    revised_draft = rev_module.apply_revisions(
                        st.session_state.current_draft,
                        clean_suggestions
                    )

                    # Update State
                    st.session_state.current_draft = revised_draft

                    # Log History
                    hist_tracker.add_revision(
                        st.session_state.version,
                        st.session_state.current_assessment,
                        clean_suggestions
                    )
                    st.session_state.version += 1

                st.success("Revisions applied! Check the 'Load & View' tab to see the updated text.")
                st.rerun()

    # --- TAB 4: FINAL REPORT ---
    with tab4:
        st.subheader("Step 4: Generate Final Report")

        if st.button("💾 Generate & Save Final Report", type="primary"):
            if not st.session_state.current_draft:
                st.error("No draft available to finalize.")
            else:
                final_path = rep_gen.generate_final_report(
                    st.session_state.current_draft,
                    st.session_state.current_assessment
                )
                st.balloons()
                st.success(f"Report saved to: {Config.FINAL_REPORT}")

                with open(Config.FINAL_REPORT, "r") as f:
                    st.download_button(
                        label="⬇️ Download Markdown Report",
                        data=f,
                        file_name="Final_Systematic_Review.md",
                        mime="text/markdown"
                    )

        if os.path.exists(Config.FINAL_REPORT):
             with st.expander("Preview Final Report"):
                with open(Config.FINAL_REPORT, "r") as f:
                    st.markdown(f.read())

if __name__ == "__main__":
    main()

Overwriting app.py


In [ ]:
# 1. Install necessary packages
!pip install -q streamlit pyngrok

# 2. Set up Ngrok (Replace the string below with your actual token)
from pyngrok import ngrok

# PASTE YOUR TOKEN BELOW inside the quotes
# Example: ngrok.set_auth_token("2FwJ...your_token_here...8Jd")
ngrok.set_auth_token("38BmkJsmTAUgy9lcRfhHisCBSM5_3AuvaxKX5HAAGQ9bVqR39")

# 3. Kill any existing streamlit processes to avoid conflicts
!pkill streamlit

# 4. Run Streamlit in the background
import subprocess
process = subprocess.Popen(['streamlit', 'run', 'app.py'])

# 5. Open the tunnel
try:
    # Open a HTTP tunnel on the default port 8501
    public_url = ngrok.connect(8501)
    print(f"🚀 Your Stable App URL: {public_url}")
except Exception as e:
    print(f"Error starting tunnel: {e}")

# Keep the cell running
import time
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("Stopping app...")
    process.terminate()
    ngrok.kill()

🚀 Your Stable App URL: NgrokTunnel: "https://porter-provable-unfabulously.ngrok-free.dev" -> "http://localhost:8501"
